# 05 — Exécution séparée des pipelines A, B et C

Ce notebook permet de lancer un pipeline à la fois sur les 500 questions. Pour le protocole final, répétez les runs `0`, `1` et `2` avec les seeds `0`, `1` et `2`. La cellule d'exécution est désactivée par défaut pour éviter un lancement accidentel coûteux.

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PIPELINE = 'A'  # choisir A, B ou C
RUN_INDEX = 0
SEED = 0
RUN_NOW = False  # passer à True uniquement quand Ollama est démarré
MODEL = 'qwen2.5:7b'
TEMPERATURE = 0.0
TOP_K = 5


In [ ]:
output = ROOT / f'results/study_run_{RUN_INDEX:02d}/pipeline_{PIPELINE}/generated_sql.json'
common = ['--questions', 'data/processed/questions.json', '--output', str(output.relative_to(ROOT)), '--model', MODEL, '--temperature', str(TEMPERATURE), '--seed', str(SEED), '--run_index', str(RUN_INDEX)]
scripts = {
    'A': ['src/pipeline_baseline.py', *common],
    'B': ['src/pipeline_schema_rag.py', '--index_dir', 'data/index', *common, '--method', 'hybrid', '--top_k', str(TOP_K)],
    'C': ['src/pipeline_business_rag.py', *common, '--top_k', str(TOP_K)],
}
command = ['python', *scripts[PIPELINE]]
print(' '.join(command))
if RUN_NOW:
    subprocess.run(command, cwd=ROOT, check=True)

In [ ]:
generated = f'results/study_run_{RUN_INDEX:02d}/pipeline_{PIPELINE}/generated_sql.json'
structural = f'results/study_run_{RUN_INDEX:02d}/pipeline_{PIPELINE}/structural_validation.json'
execution = f'results/study_run_{RUN_INDEX:02d}/pipeline_{PIPELINE}/execution_accuracy.json'
validation_command = ['python', 'src/sql_validator.py', 'validate-batch', '--generated', generated, '--schemas_dir', 'data/schemas', '--output', structural]
accuracy_command = ['python', 'src/execution_accuracy.py', '--generated', generated, '--questions', 'data/processed/questions.json', '--databases_dir', 'data/raw/bird/dev_databases', '--output', execution]
print('Validation :', ' '.join(validation_command))
print('Execution Accuracy :', ' '.join(accuracy_command))
if RUN_NOW:
    subprocess.run(validation_command, cwd=ROOT, check=True)
    subprocess.run(accuracy_command, cwd=ROOT, check=True)

## Protocole

Exécutez A, puis B, puis C ; ne modifiez ni le modèle, ni la température, ni `top_k` entre les pipelines. Répétez ensuite avec `RUN_INDEX = SEED = 1`, puis `2`. Le pipeline D est traité dans le notebook suivant.